# Projeto Olist E-commerce
## 07. Análise Exploratória de Dados com Python

### Objetivo
Explorar as camadas analíticas construídas no PostgreSQL, identificar padrões, relações e oportunidades de negócio e selecionar os principais indicadores e insights para o dashboard.

In [10]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

## 7.1 Conexão com o PostgreSQL

Nesta etapa será criada a conexão entre o Python e o banco PostgreSQL
para permitir o carregamento das views analíticas construídas no SQL.

In [12]:
from getpass import getpass
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

senha = getpass("Digite a senha do PostgreSQL: ")
url = URL.create("postgresql+psycopg2",
    username="postgres",
    password=senha,
    host="localhost",
    port=5432,
    database="olist_ecommerce")
engine = create_engine(url)

Digite a senha do PostgreSQL:  ········


## 7.2 Carregamento das bases analíticas

As views criadas no PostgreSQL serão carregadas como DataFrames do pandas.
Cada DataFrame possui uma granularidade específica e será utilizado
nas diferentes análises exploratórias.

In [14]:
orders = pd.read_sql("SELECT * FROM modelado.vw_orders_analytics", engine)
customers = pd.read_sql("SELECT * FROM modelado.vw_customers_analytics", engine)
products = pd.read_sql("SELECT * FROM modelado.vw_products_analytics", engine)
sellers = pd.read_sql("SELECT * FROM modelado.vw_sellers_analytics", engine)
categories = pd.read_sql("SELECT * FROM modelado.vw_categories_analytics", engine)
geolocation = pd.read_sql("SELECT * FROM modelado.vw_geolocation_zip", engine)
calendar = pd.read_sql("SELECT * FROM modelado.dim_calendar", engine)

In [15]:
print("orders:", orders.shape)
print("customers:", customers.shape)
print("products:", products.shape)
print("sellers:", sellers.shape)
print("categories:", categories.shape)
print("geolocation:", geolocation.shape)
print("calendar:", calendar.shape)

orders: (99441, 37)
customers: (96096, 11)
products: (32951, 19)
sellers: (3095, 14)
categories: (74, 14)
geolocation: (19015, 6)
calendar: (799, 11)


## 7.3 Validação inicial dos DataFrames
Nesta etapa serão verificados os tipos de dados, valores ausentes e estatísticas básicas das principais bases analíticas antes do início da exploração.

In [16]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 37 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   order_id                        99441 non-null  str           
 1   customer_id                     99441 non-null  str           
 2   customer_unique_id              99441 non-null  str           
 3   order_status                    99441 non-null  str           
 4   pedido_valido                   99441 non-null  bool          
 5   order_purchase_timestamp        99441 non-null  datetime64[us]
 6   order_approved_at               99281 non-null  datetime64[us]
 7   order_delivered_carrier_date    97658 non-null  datetime64[us]
 8   order_delivered_customer_date   96476 non-null  datetime64[us]
 9   order_estimated_delivery_date   99441 non-null  datetime64[us]
 10  customer_zip_code_prefix        99441 non-null  str           
 11  customer_city

In [17]:
orders.isna().sum().sort_values(ascending=False)

entregue_no_prazo                 2965
dias_para_entrega                 2965
diferenca_prazo_dias              2965
dias_atraso                       2965
order_delivered_customer_date     2965
order_delivered_carrier_date      1783
nota_media                         768
maior_nota                         768
menor_nota                         768
order_approved_at                  160
possui_pagamento_valor_zero          1
formas_pagamento                     1
possui_parcelamento_zero             1
total_pago                           1
maior_numero_parcelas                1
diferenca_pagamento_pedido           1
pedido_valido                        0
order_id                             0
order_status                         0
customer_unique_id                   0
customer_id                          0
valor_produtos                       0
quantidade_sellers                   0
quantidade_produtos                  0
quantidade_itens                     0
order_estimated_delivery_

In [18]:
orders.describe().T

,count,mean,min,25%,50%,75%,max,std
order_purchase_timestamp,99441,2017-12-31 08:43:12.776581,2016-09-04 21:15:19,2017-09-12 14:46:19,2018-01-18 23:04:36,2018-05-04 15:42:16,2018-10-17 17:30:18,NaN
order_approved_at,99281,2017-12-31 18:35:24.098800,2016-09-15 12:16:38,2017-09-12 23:24:16,2018-01-19 11:36:13,2018-05-04 20:35:10,2018-09-03 17:40:06,NaN
order_delivered_carrier_date,97658,2018-01-04 21:49:48.138278,2016-10-08 10:34:01,2017-09-15 22:28:50.250000,2018-01-24 16:10:58,2018-05-08 13:37:45,2018-09-11 19:48:28,NaN
order_delivered_customer_date,96476,2018-01-14 12:09:19.035542,2016-10-11 13:46:32,2017-09-25 22:07:22.250000,2018-02-02 19:28:10.500000,2018-05-15 22:48:52.250000,2018-10-17 13:22:46,NaN
order_estimated_delivery_date,99441,2018-01-24 03:08:37.730111,2016-09-30 00:00:00,2017-10-03 00:00:00,2018-02-15 00:00:00,2018-05-25 00:00:00,2018-11-12 00:00:00,NaN
dias_para_entrega,96476.0,12.558693,0.53,6.77,10.22,15.72,209.63,9.54653
diferenca_prazo_dias,96476.0,-11.179121,-146.02,-16.24,-11.95,-6.39,188.98,10.186118
dias_atraso,96476.0,0.774959,0.0,0.0,0.0,0.0,188.98,4.753109
quantidade_itens,99441.0,1.132833,0.0,1.0,1.0,1.0,21.0,0.545666
quantidade_produtos,99441.0,1.030008,0.0,1.0,1.0,1.0,8.0,0.243344


In [19]:
orders.describe().T

,count,mean,min,25%,50%,75%,max,std
order_purchase_timestamp,99441,2017-12-31 08:43:12.776581,2016-09-04 21:15:19,2017-09-12 14:46:19,2018-01-18 23:04:36,2018-05-04 15:42:16,2018-10-17 17:30:18,NaN
order_approved_at,99281,2017-12-31 18:35:24.098800,2016-09-15 12:16:38,2017-09-12 23:24:16,2018-01-19 11:36:13,2018-05-04 20:35:10,2018-09-03 17:40:06,NaN
order_delivered_carrier_date,97658,2018-01-04 21:49:48.138278,2016-10-08 10:34:01,2017-09-15 22:28:50.250000,2018-01-24 16:10:58,2018-05-08 13:37:45,2018-09-11 19:48:28,NaN
order_delivered_customer_date,96476,2018-01-14 12:09:19.035542,2016-10-11 13:46:32,2017-09-25 22:07:22.250000,2018-02-02 19:28:10.500000,2018-05-15 22:48:52.250000,2018-10-17 13:22:46,NaN
order_estimated_delivery_date,99441,2018-01-24 03:08:37.730111,2016-09-30 00:00:00,2017-10-03 00:00:00,2018-02-15 00:00:00,2018-05-25 00:00:00,2018-11-12 00:00:00,NaN
dias_para_entrega,96476.0,12.558693,0.53,6.77,10.22,15.72,209.63,9.54653
diferenca_prazo_dias,96476.0,-11.179121,-146.02,-16.24,-11.95,-6.39,188.98,10.186118
dias_atraso,96476.0,0.774959,0.0,0.0,0.0,0.0,188.98,4.753109
quantidade_itens,99441.0,1.132833,0.0,1.0,1.0,1.0,21.0,0.545666
quantidade_produtos,99441.0,1.030008,0.0,1.0,1.0,1.0,8.0,0.243344


In [20]:
bases = {
    "orders": orders,
    "customers": customers,
    "products": products,
    "sellers": sellers,
    "categories": categories,
    "geolocation": geolocation,
    "calendar": calendar}
for nome, base in bases.items():
    print(f"{nome}: {base.shape[0]} linhas | {base.shape[1]} colunas | {base.duplicated().sum()} duplicidades exatas")

orders: 99441 linhas | 37 colunas | 0 duplicidades exatas
customers: 96096 linhas | 11 colunas | 0 duplicidades exatas
products: 32951 linhas | 19 colunas | 0 duplicidades exatas
sellers: 3095 linhas | 14 colunas | 0 duplicidades exatas
categories: 74 linhas | 14 colunas | 0 duplicidades exatas
geolocation: 19015 linhas | 6 colunas | 0 duplicidades exatas
calendar: 799 linhas | 11 colunas | 0 duplicidades exatas


## 7.4 Visão geral do negócio
Nesta etapa serão calculados os principais indicadores gerais do e-commerce para construir uma visão inicial da operação antes das análises aprofundadas.

In [58]:
orders_validos = orders[orders["pedido_valido"] == True]

entregas_avaliaveis = orders_validos[orders_validos["entregue_no_prazo"].notna()]

total_pedidos = orders_validos["order_id"].nunique()
total_clientes = orders_validos["customer_unique_id"].nunique()
valor_total = orders_validos["valor_pedido"].sum()
ticket_medio = orders_validos["valor_pedido"].mean()
nota_media = orders_validos["nota_media"].mean()
prazo_medio = orders_validos["dias_para_entrega"].mean()

percentual_atraso = ((entregas_avaliaveis["entregue_no_prazo"] == False).mean() * 100)

total_itens = orders_validos["quantidade_itens"].sum()
produtos_vendidos = products.loc[products["itens_vendidos"] > 0, "product_id"].nunique()
sellers_ativos = sellers.loc[sellers["pedidos_validos"] > 0, "seller_id"].nunique()
frete_total = orders_validos["valor_frete"].sum()
percentual_frete = (frete_total / valor_total) * 100

print(f"Pedidos válidos: {total_pedidos:,}")
print(f"Clientes únicos: {total_clientes:,}")
print(f"Valor total movimentado: R$ {valor_total:,.2f}")
print(f"Ticket médio: R$ {ticket_medio:,.2f}")
print(f"Nota média: {nota_media:.2f}")
print(f"Prazo médio de entrega: {prazo_medio:.2f} dias")
print(f"Entregas avaliáveis: {len(entregas_avaliaveis):,}")
print(f"Pedidos atrasados: {percentual_atraso:.2f}%")
print(f"Itens vendidos: {total_itens:,.0f}")
print(f"Produtos vendidos: {produtos_vendidos:,}")
print(f"Sellers ativos: {sellers_ativos:,}")
print(f"Frete total: R$ {frete_total:,.2f}")
print(f"Participação do frete: {percentual_frete:.2f}%")

Pedidos válidos: 98,207
Clientes únicos: 94,990
Valor total movimentado: R$ 15,735,527.03
Ticket médio: R$ 160.23
Nota média: 4.12
Prazo médio de entrega: 12.56 dias
Entregas avaliáveis: 96,470
Pedidos atrasados: 8.11%
Itens vendidos: 112,101
Produtos vendidos: 32,729
Sellers ativos: 3,053
Frete total: R$ 2,241,126.29
Participação do frete: 14.24%


### Conclusão
A base possui 98.207 pedidos considerados válidos, realizados por 94.990 clientes únicos. O valor total movimentado foi de BRL 15,7 milhões, com ticket médio de BRL 160,23.

A avaliação média foi de 4,12 pontos e o prazo médio de entrega foi de 12,56 dias. Entre as 96.470 entregas com informação suficiente para avaliação do prazo, 8,11% foram entregues após a data estimada.

Foram vendidos 112.101 itens, envolvendo 32.729 produtos e 3.053 sellers ativos. O frete somou BRL 2,24 milhões e representou 14,24% do valor movimentado.

## 7.5 Evolução temporal
Nesta etapa será analisada a evolução mensal dos pedidos, clientes, valor movimentado e ticket médio. A análise considera que setembro de 2016 e setembro de 2018 possuem cobertura parcial.

### 7.5.1  Cobertura temporal 

In [22]:
data_inicial = orders_validos["order_purchase_timestamp"].min()
data_final = orders_validos["order_purchase_timestamp"].max()

print(f"Primeira compra: {data_inicial}")
print(f"Última compra: {data_final}")

Primeira compra: 2016-09-04 21:15:19
Última compra: 2018-09-03 09:06:57


In [23]:
%pip install plotly

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ------------------- -------------------- 4.7/9.7 MB 45.6 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 41.1 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [24]:
import plotly.express as px
import plotly.graph_objects as go

In [34]:
import plotly.io as pio
pio.renderers.default = "iframe"

In [59]:
# 7.5.2 Preparação da base mensal
orders_validos = orders_validos.copy()
orders_validos["order_purchase_timestamp"] = pd.to_datetime(orders_validos["order_purchase_timestamp"])
orders_validos["mes"] = orders_validos["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()

evolucao_mensal = orders_validos.groupby("mes", as_index=False).agg(pedidos=("order_id","nunique"),
clientes=("customer_unique_id","nunique"),valor_movimentado=("valor_pedido","sum"),ticket_medio=("valor_pedido","mean"))
evolucao_mensal_completa = evolucao_mensal[(evolucao_mensal["mes"] >= "2016-10-01")&(evolucao_mensal["mes"] <= "2018-08-01")].copy()

print(f"Base mensal criada: {len(evolucao_mensal)} meses")
print(f"Meses completos utilizados: {len(evolucao_mensal_completa)}")

Base mensal criada: 24 meses
Meses completos utilizados: 22


In [60]:
# 7.5.3 Evolução mensal dos pedidos
fig = px.area(evolucao_mensal_completa,x="mes",y="pedidos",markers=True,title="Evolução mensal dos pedidos")
fig.update_traces(text=evolucao_mensal_completa["pedidos"],texttemplate="%{text:,.0f}",textposition="top center")
fig.update_layout(xaxis_title="Mês",yaxis_title="Pedidos",hovermode="x unified")
fig.show()

### Conclusão
O volume de pedidos apresentou forte crescimento ao longo de 2017, atingindo seu maior nível no final do ano. Em 2018, a operação passou a apresentar um comportamento mais estável, mantendo aproximadamente 6 mil a 7 mil pedidos por mês.

O resultado indica uma fase inicial de expansão seguida por estabilização em um patamar mais elevado de demanda.

In [61]:
# 7.5.4 Valor movimentado por mês
fig = px.bar(evolucao_mensal_completa,x="mes",y="valor_movimentado",title="Valor movimentado por mês")
fig.update_traces(hovertemplate="Mês: %{x|%m/%Y}<br>Valor: BRL %{y:,.2f}<extra></extra>")
for _, linha in evolucao_mensal_completa.iterrows():fig.add_annotation(x=linha["mes"],y=linha["valor_movimentado"],
text=f"BRL {linha['valor_movimentado']:,.0f}",showarrow=False,yshift=12,font=dict(size=10))
fig.update_layout(xaxis_title="Mês",yaxis_title="Valor movimentado",height=650,margin=dict(t=100))
fig.show()

### Conclusão
O valor movimentado apresentou forte crescimento ao longo de 2017, acompanhando a expansão do volume de pedidos. Novembro de 2017 registrou o maior resultado do período, com aproximadamente BRL 1,17 milhão.

Após uma queda em dezembro, o valor voltou a crescer no início de 2018 e permaneceu relativamente estável em um patamar próximo ou superior a BRL 1 milhão por mês. A partir de junho de 2018 observa-se uma leve redução, mas ainda em níveis significativamente superiores aos registrados no início da série.

In [62]:
# 7.5.5 Evolução do ticket médio
ticket_mensal=evolucao_mensal[(evolucao_mensal["mes"]>="2017-01-01")&(evolucao_mensal["mes"]<="2018-08-01")].copy()
fig=px.line(ticket_mensal,x="mes",y="ticket_medio",markers=True,title="Evolução do ticket médio")
fig.update_traces(text=ticket_mensal["ticket_medio"],texttemplate="BRL %{text:.0f}",textposition="top center",textfont=dict(size=14),hovertemplate="Mês: %{x|%m/%Y}<br>Ticket médio: BRL %{y:,.2f}<extra></extra>")
fig.update_layout(xaxis_title="Mês",yaxis_title="Ticket médio")
fig.show()

### Conclusão
O ticket médio permaneceu relativamente estável entre janeiro de 2017 e agosto de 2018, variando aproximadamente entre BRL 147 e BRL 174.

Apesar de algumas oscilações mensais, não há uma tendência sustentada de crescimento do valor médio por pedido. Isso reforça que o aumento do valor movimentado no período foi impulsionado principalmente pelo crescimento do volume de pedidos.

In [63]:
# 7.5.6 Crescimento mensal dos pedidos
crescimento=evolucao_mensal[(evolucao_mensal["mes"]>="2017-01-01")&(evolucao_mensal["mes"]<="2018-08-01")].copy()
crescimento["crescimento_pct"]=crescimento["pedidos"].pct_change()*100
crescimento=crescimento.dropna(subset=["crescimento_pct"])
crescimento["resultado"]=np.where(crescimento["crescimento_pct"]>=0,"Crescimento","Queda")
fig=px.bar(crescimento,x="mes",y="crescimento_pct",color="resultado",text="crescimento_pct",title="Variação mensal dos pedidos",labels={"mes":"Mês","crescimento_pct":"Variação (%)","resultado":"Resultado"})
fig.update_traces(texttemplate="%{text:.1f}%",textposition="outside",cliponaxis=False,hovertemplate="Mês: %{x|%m/%Y}<br>Variação: %{y:.1f}%<extra></extra>")
fig.update_layout(xaxis_tickformat="%b/%Y",xaxis_tickangle=-45,yaxis_title="Variação (%)",bargap=0.25)
fig.add_hline(y=0)
fig.show()

### Conclusão
A evolução mensal mostra uma fase de forte expansão ao longo de 2017, acompanhada por oscilações relevantes no volume de pedidos. Fevereiro registrou crescimento de 118,3% sobre janeiro, enquanto novembro apresentou outro salto expressivo de 63,3%.

Após o pico de novembro, dezembro apresentou retração de 24,3%, seguida por recuperação de 27,9% em janeiro de 2018.

Ao longo de 2018, as variações mensais ficaram consideravelmente menores, indicando uma operação mais estável em comparação com o período de expansão observado em 2017.

In [64]:
# 7.5.7 Distribuição mensal dos pedidos por ano
sazonalidade=evolucao_mensal_completa.copy()
sazonalidade["ano"]=sazonalidade["mes"].dt.year.astype(str)
meses=["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"]
sazonalidade["mes_nome"]=sazonalidade["mes"].dt.month.map(dict(enumerate(meses,start=1)))
heatmap=sazonalidade.pivot(index="ano",columns="mes_nome",values="pedidos").reindex(columns=meses)
fig=px.imshow(heatmap,text_auto=".0f",aspect="auto",labels={"x":"Mês","y":"Ano","color":"Pedidos"},title="Distribuição mensal dos pedidos por ano")
fig.update_xaxes(type="category")
fig.update_yaxes(type="category")
fig.show()

### Conclusão
A distribuição mensal reforça a expansão observada ao longo de 2017, com novembro registrando o maior volume do ano, com 7.423 pedidos.

Em 2018, o volume mensal permaneceu em um patamar mais elevado, variando aproximadamente entre 6,1 mil e 7,2 mil pedidos entre janeiro e agosto.

Como somente 2017 possui cobertura completa dos 12 meses, os dados não são suficientes para confirmar um padrão sazonal recorrente. O gráfico deve ser utilizado principalmente para comparar a evolução mensal e os diferentes níveis de volume ao longo do período disponível.

In [65]:
# 7.5.8 Faturamento por região
regioes={"AC":"Norte","AP":"Norte","AM":"Norte","PA":"Norte","RO":"Norte","RR":"Norte","TO":"Norte","AL":"Nordeste","BA":"Nordeste","CE":"Nordeste","MA":"Nordeste","PB":"Nordeste","PE":"Nordeste","PI":"Nordeste","RN":"Nordeste","SE":"Nordeste","DF":"Centro-Oeste","GO":"Centro-Oeste","MT":"Centro-Oeste","MS":"Centro-Oeste","ES":"Sudeste","MG":"Sudeste","RJ":"Sudeste","SP":"Sudeste","PR":"Sul","RS":"Sul","SC":"Sul"}
orders_validos["regiao"]=orders_validos["customer_state"].map(regioes)
faturamento_regiao=orders_validos.groupby("regiao",as_index=False).agg(faturamento=("valor_produtos","sum"),pedidos=("order_id","nunique"))
faturamento_regiao=faturamento_regiao.sort_values("faturamento",ascending=True)
fig=px.bar(faturamento_regiao,x="faturamento",y="regiao",orientation="h",text="faturamento",title="Faturamento por região",labels={"faturamento":"Faturamento","regiao":"Região"})
fig.update_traces(texttemplate="BRL %{text:,.0f}",textposition="outside",cliponaxis=False,hovertemplate="Região: %{y}<br>Faturamento: BRL %{x:,.2f}<extra></extra>")
fig.update_layout(xaxis_title="Faturamento",yaxis_title="Região",showlegend=False)
fig.show()

### Critério de análise
O faturamento por região considera todo o período disponível na base e todos os pedidos considerados válidos, excluindo apenas os status `canceled` e `unavailable`.

O faturamento corresponde à soma do valor dos produtos, sem incluir o frete. Meses parciais presentes na base foram mantidos para preservar integralmente os registros disponíveis.

In [68]:
# 7.5.9 Comparativo de faturamento por região e ano
faturamento_ano_regiao=orders_validos.copy()
faturamento_ano_regiao["ano"]=faturamento_ano_regiao["order_purchase_timestamp"].dt.year.astype(str)
faturamento_ano_regiao["regiao"]=faturamento_ano_regiao["customer_state"].map(regioes)
faturamento_ano_regiao=faturamento_ano_regiao.groupby(["regiao","ano"],as_index=False).agg(faturamento=("valor_produtos","sum"))
ordem_regioes=["Sudeste","Sul","Nordeste","Centro-Oeste","Norte"]
faturamento_ano_regiao["regiao"]=pd.Categorical(faturamento_ano_regiao["regiao"],categories=ordem_regioes,ordered=True)
faturamento_ano_regiao=faturamento_ano_regiao.sort_values(["regiao","ano"])
fig=px.bar(faturamento_ano_regiao,x="faturamento",y="regiao",color="ano",barmode="group",orientation="h",text="faturamento",title="Comparativo de faturamento por região e ano",labels={"faturamento":"Faturamento","regiao":"Região","ano":"Ano"})
fig.update_traces(texttemplate="BRL %{text:,.0f}",textposition="outside",cliponaxis=False,hovertemplate="Região: %{y}<br>Ano: %{fullData.name}<br>Faturamento: BRL %{x:,.2f}<extra></extra>")
fig.update_layout(xaxis_title="Faturamento",yaxis_title="Região",legend_title="Ano",height=650,bargap=0.2,bargroupgap=0.05)
fig.show()

### Observação
Os anos de 2016 e 2018 possuem cobertura parcial na base. Portanto, os valores não representam anos completos e não devem ser utilizados isoladamente para calcular crescimento anual. A comparação mostra o faturamento registrado no período disponível de cada ano.
    

### Conclusão

- **O negócio ganhou escala ao longo de 2017.** O volume mensal saiu de menos de 1 mil pedidos no início do ano e passou para mais de 4 mil no segundo semestre. Em 2018, a operação se manteve em um patamar mais alto, geralmente entre 6 mil e 7 mil pedidos por mês.

- **Novembro de 2017 foi o principal destaque.** Foram 7.423 pedidos e aproximadamente BRL 1,17 milhão movimentado. O crescimento mensal de pedidos chegou a 63,3%. Esse pico merece uma análise específica para entender quais clientes, categorias ou outros fatores sustentaram esse resultado.

- **O aumento do valor movimentado veio principalmente do volume.** O ticket médio ficou relativamente estável, aproximadamente entre BRL 147 e BRL 174 no período analisado de 2017 a 2018. Isso indica que o crescimento ocorreu mais pelo aumento da quantidade de pedidos do que pelo aumento do valor médio de cada compra.

- **O Sudeste concentra grande parte do faturamento.** Considerando o valor dos produtos e todos os pedidos válidos, a região movimentou aproximadamente BRL 8,82 milhões, muito acima de Sul, Nordeste, Centro-Oeste e Norte. Vale aprofundar essa concentração por estado.

- **O frete tem peso relevante na operação.** Foram aproximadamente BRL 2,24 milhões em frete, equivalentes a 14,24% do valor total movimentado. Esse resultado justifica análises posteriores por região, categoria e valor do pedido.

- **A experiência geral é positiva, mas a logística merece atenção.** A nota média foi 4,12 e, entre as 96.470 entregas que puderam ser avaliadas, 8,11% ocorreram após a data estimada. O próximo passo é verificar quanto o atraso está associado à queda das avaliações.

- **Os períodos precisam ser comparados com cuidado.** A base começa em setembro de 2016 e termina em setembro de 2018. Por isso, 2016 e 2018 não representam anos completos. Os registros foram mantidos para preservar toda a informação disponível, mas os totais anuais não devem ser comparados como se os três anos tivessem a mesma cobertura.

**Síntese:** os dados mostram uma operação que passou por forte expansão em 2017 e chegou a 2018 em um nível de demanda bem mais elevado. Até aqui, o crescimento parece estar muito mais relacionado ao aumento da quantidade de pedidos do que ao aumento do ticket médio, com forte concentração comercial no Sudeste.